# Brainrot Translation Quality Classifier Training Notebook

This notebook trains the local good/bad translation classifier used after the FLAN-T5 translator.

Expected uploaded CSV:

- `translation_quality_classifier_dataset.csv`

Expected columns:

- `source_text`
- `candidate_translation`
- `label`
- `label_text`
- `reason`
- `source_row_id`

After training, put the downloaded model folder into your project at:

`models/brainrot-quality-classifier-v1`

In [ ]:
!pip -q install -U "transformers>=4.41,<5" "huggingface-hub>=0.34,<1.0" torch tqdm safetensors

In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [ ]:
import transformers
import torch

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Colab: Runtime > Change runtime type > T4 GPU.")
print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files

uploaded = files.upload()
DATASET_PATH = next(iter(uploaded.keys()))
print("dataset:", DATASET_PATH)

In [ ]:
import pandas as pd

REQUIRED_COLUMNS = ["source_text", "candidate_translation", "label", "label_text", "reason", "source_row_id"]

df = pd.read_csv(DATASET_PATH)
missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["source_text", "candidate_translation", "label"])
df["source_text"] = df["source_text"].astype(str).str.strip()
df["candidate_translation"] = df["candidate_translation"].astype(str).str.strip()
df = df[df["source_text"].ne("") & df["candidate_translation"].ne("")]
df["label"] = df["label"].astype(int)

bad_labels = sorted(set(df["label"]) - {0, 1})
if bad_labels:
    raise ValueError(f"Labels must be 0 or 1 only. Found: {bad_labels}")
if len(df) < 1000:
    raise ValueError(f"Classifier dataset is too small: {len(df)} rows")

print("rows, columns:", df.shape)
print("label counts:")
print(df["label_text"].value_counts())
display(df.sample(min(20, len(df)), random_state=42))

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

BASE_MODEL = "distilbert-base-uncased"
OUTPUT_DIR = "brainrot-quality-classifier-v1"
MAX_LENGTH = 192
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
EVAL_RATIO = 0.15
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

rows = []
for _, row in df.iterrows():
    rows.append({
        "text": (
            f"Source brainrot text: {row['source_text']}\n"
            f"Candidate normal English translation: {row['candidate_translation']}"
        ),
        "label": int(row["label"]),
        "source_row_id": str(row.get("source_row_id") or row["source_text"]),
    })

grouped = {}
for row in rows:
    grouped.setdefault(row["source_row_id"], []).append(row)

groups = list(grouped.values())
random.shuffle(groups)
eval_target = max(1, int(len(rows) * EVAL_RATIO))
train_rows = []
eval_rows = []
for group in groups:
    if len(eval_rows) < eval_target:
        eval_rows.extend(group)
    else:
        train_rows.extend(group)

random.shuffle(train_rows)
random.shuffle(eval_rows)
print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))

In [ ]:
class QualityDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "bad_translation", 1: "good_translation"},
    label2id={"bad_translation": 0, "good_translation": 1},
)
model.to(device)

def collate(rows):
    encoded = tokenizer(
        [row["text"] for row in rows],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor([int(row["label"]) for row in rows], dtype=torch.long)
    return {key: value.to(device) for key, value in encoded.items()}

train_loader = DataLoader(QualityDataset(train_rows), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
eval_loader = DataLoader(QualityDataset(eval_rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps,
)

In [ ]:
def evaluate():
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in eval_loader:
            outputs = model(**batch)
            total_loss += float(outputs.loss.detach().cpu().item())
            predictions = outputs.logits.argmax(dim=-1)
            correct += int((predictions == batch["labels"]).sum().detach().cpu().item())
            total += int(batch["labels"].numel())
    model.train()
    return {
        "eval_loss": total_loss / max(1, len(eval_loader)),
        "eval_accuracy": correct / max(1, total),
    }


history = []
model.train()
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(progress, start=1):
        outputs = model(**batch)
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch}, step {step}: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += float(loss.detach().cpu().item())
        if step % 25 == 0:
            progress.set_postfix(train_loss=running_loss / step)

    metrics = evaluate()
    metrics["epoch"] = epoch
    metrics["train_loss"] = running_loss / max(1, len(train_loader))
    history.append(metrics)
    print(
        f"epoch={epoch} train_loss={metrics['train_loss']:.4f} "
        f"eval_loss={metrics['eval_loss']:.4f} eval_accuracy={metrics['eval_accuracy']:.4f}"
    )

history

In [ ]:
import json
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

report = {
    "base_model": BASE_MODEL,
    "train_rows": len(train_rows),
    "eval_rows": len(eval_rows),
    "history": history,
}
(output_dir / "quality_classifier_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("saved classifier:", output_dir)

In [ ]:
def score_quality(source_text, candidate_translation):
    text = (
        f"Source brainrot text: {source_text}\n"
        f"Candidate normal English translation: {candidate_translation}"
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    probabilities = torch.softmax(logits, dim=-1)[0]
    return float(probabilities[1].detach().cpu().item())


smoke_tests = [
    ("he has rizz", "He has charm or romantic appeal."),
    ("he has rizz", "He has rizz."),
    ("skill issue", "The problem is caused by lack of ability."),
    ("skill issue", "This means charm, charisma, or romantic appeal."),
]

for source_text, candidate in smoke_tests:
    print(f"{score_quality(source_text, candidate):.4f} :: {source_text} => {candidate}")

In [ ]:
!rm -f brainrot-quality-classifier-v1.zip
!zip -qr brainrot-quality-classifier-v1.zip brainrot-quality-classifier-v1
files.download("brainrot-quality-classifier-v1.zip")